In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve, schur
import scipy as sp
from scipy.integrate import solve_ivp
import sys, os, pickle
from joblib import Parallel, delayed, cpu_count
from matplotlib.animation import FuncAnimation
from utility import orbit, call_method
from models import Mckean_Vlasov
import datetime

In [ ]:
if __name__ == "__main__":
    param_file = "./config_models/kuramoto_param.in"  # JSON file containing model parameters
    model = Mckean_Vlasov(param_file)
    # print("Loaded parameters:", model.n_z)

    model.n_z = 100
    model.xmin = -np.pi
    model.xmax = np.pi
    model.update_params() #(**{'n_z':500})


    f = model.dydt_per
    J = model.jacobian_per

    z, z_centers, h = model.mesh1D  # Get the mesh and centers
    
    # model.m0 = float(h*np.ones_like(y0) @ y0)
#     SOL = []
#     #Let's scan diferent values of alpha_shift and see what happens before and after I_c = 2*sec(alpha_shift)
#     Alpha = [np.pi/4,np.pi]#, np.pi/3]
#     fig, axes = plt.subplots(len(Alpha), 4, figsize=(12, 10))
#     for i, alpha in enumerate(Alpha):
#         y0 = (1/(2*np.pi))*np.ones_like(z_centers) + 1e-5*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
#         model.alpha_shift = alpha
#         I_c = 2/np.cos(model.alpha_shift)
        
        
#         for k, I in enumerate([0.5*I_c, 0.9*I_c, 1.1*I_c, 2*I_c]):
#             model.I = I
#     #start from a gaussian
    
            

#     # sol0 = solve_ivp(f, (0, 10*model.T_ini), y0, method='RK45',
#     #              rtol=1e-7, atol=1e-9,
#     #              t_eval=np.linspace(0, 10*model.T_ini, 100))
#             sol = solve_ivp(f, (0, 4*model.T_ini), y0, method='BDF', jac = J, 
#                     rtol=1e-4, atol=1e-6,
#                     t_eval=np.linspace(0,4*model.T_ini, 300))
#             # first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
            
#             axes[i, k].plot(sol.t, sol.y[0,:])
#             axes[i, k].set_title(f"alpha={alpha:.2f}, I={I:.2f}")
#             axes[i, k].set_xlabel("Time")
#             axes[i, k].set_ylabel(r"$\int_{\Omega} z \rho(z,t)dz $")
#             SOL.append((alpha, I, sol))
#             #update y0 for the next run
#             # y0 = np.mean(sol.y, axis=1)
#         #     model.m0 = float(h*np.ones_like(y0) @ y0)
#     plt.suptitle("First moment evolution for different alpha and I values")
    
#     #share labels
# #     for ax in axes.flat:
# #         ax.label_outer()
#     plt.tight_layout()
#     plt.show()

In [ ]:
#test of the convolution product
kernel = lambda z: model.kuramoto_potential(z)
# kernel = lambda z: np.where( np.abs(z) <= 1e-8, 1, 0)/h
z = np.linspace(0,model.xmax - model.xmin, model.n_z-1, endpoint=False)
# def kernel(z):
#     #A squared kernel to test the convolution
#     r = np.where((z >= -np.pi/20) & (z <= np.pi/20), 1, 0)
#     return r
model.alpha_shift = 0.0

C = model.C_mat_per
K = kernel(z_centers - z_centers[0]) #kernel evaluated at the centers
y0 = np.exp(z_centers)
# y0 = np.where((z_centers >= -np.pi/40) & (z_centers <= np.pi/40), 1, 0)
anal_convolv = -np.pi*np.cos(z_centers - model.alpha_shift)
conv_result = sp.signal.convolve(K,y0, mode='same')*h
# C_shit = np.fft.fftshift(C)
conv_with_mat = C @ y0 #/(-np.pi)#np.fft.fftshift(h*C @ y0)
#shifted convolution using FFT
convol_rest_ifft = h*np.fft.ifft(np.fft.fft(K)*np.fft.fft(y0)).real
# conv_fft_shifted = np.fft.fftshift(convol_rest_ifft)
plt.figure()
# plt.plot(z_centers, conv_result, label='Convolution Result')
# plt.plot(z, convol_rest_ifft, label='Convolution Result (IFFT)', linestyle='--')
plt.plot(z_centers, convol_rest_ifft, label='Convolution Result (IFFT Shifted)', linestyle='--')
# plt.plot(z_centers, anal_convolv, label='Analytical Convolution', linestyle='--')
# plt.plot(z, K, label='Kuramoto Potential', linestyle='--')
plt.plot(z_centers, conv_with_mat, label='Convolution with Matrix', linestyle='--')
plt.plot(z_centers, y0, label='y0', linestyle='--')
plt.plot(z_centers, convol_rest_ifft/conv_with_mat, label='Ratio', linestyle='--')
# plt.plot(z, K, label='Kuramoto Potential', linestyle='--')
plt.title('Convolution of y0 with Kuramoto Potential')
plt.xlabel('z')
plt.ylabel('Value')
plt.legend()
plt.show()
# print("Difference between convolution methods:", np.linalg.norm(anal_convolv - conv_fft_shifted) , np.linalg.norm(conv_result - convol_rest_ifft), np.linalg.norm(conv_with_mat - conv_fft_shifted))

In [ ]:
# for k , (alpha, I, sol) in enumerate(SOL):
#     if k!=0:
#         break
# from turtle import update

denom = 4
alpha = np.pi/denom
Ic = 2/np.cos(alpha)
y0 = (1/(2*np.pi))*np.ones_like(z_centers)+0.001*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
model.alpha_shift = alpha

I_values = [0.5*Ic, 0.9*Ic, 1.1*Ic, 2*Ic, 5*Ic]
coeff_labels = ["0.5*I_c", "0.9*I_c", "1.1*I_c", "2*I_c", "5*I_c"]
fig, axes = plt.subplots(1, 5, figsize=(14, 4), sharex=True, sharey=True)
sols = []
lines = []

for ax, I, coeff in zip(axes, I_values, coeff_labels):
    model.I = I
    sol = solve_ivp(f, (0, 20*model.T_ini), y0, method='BDF',
                    rtol=1e-7, atol=1e-9,
                    t_eval=np.linspace(0, 20*model.T_ini, 1000))
    sols.append(sol)

    line, = ax.plot(z_centers, sol.y[:, 0], label=rf"$\alpha = \dfrac{{\pi}}{{{denom}}},\ I={coeff}$")
    lines.append(line)
    ax.set_xlim(z_centers[0], z_centers[-1])
    ax.set_ylim(0.8 * np.min(sol.y), np.max(sol.y) * 1.1)
    ax.set_xlabel('z')
    ax.set_ylabel(r'$\rho(z)$')
    ax.set_title(rf'$I={coeff}$')
    ax.legend(loc='upper right')

fig.suptitle(rf'Evolution of $\rho(z,t)$ for different values of $I$ with $\alpha = \dfrac{{\pi}}{{{denom}}}, I_c = 2 \times sec(\alpha)$', fontsize=16)
fig.tight_layout()


def update(frame):
    for ax, line, sol, coeff in zip(axes, lines, sols, coeff_labels):
        line.set_ydata(sol.y[:, frame])
        ax.set_title(rf'$I={coeff},\ t={sol.t[frame]:.2f}$')

    return lines

ani = FuncAnimation(fig, update, frames=len(sols[0].t), blit=True, interval=50) 
ani.save(f'solution_with_kuramoto_potential_alpha_{alpha:.2f}_bdf.gif', writer='imagemagick')
plt.show()
   


In [ ]:
denom = 4
alpha = np.pi/denom
Ic = 2/np.cos(alpha)
y0 = (1/(2*np.pi))*np.ones_like(z_centers)+0.001*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
# y0 = sol.y[:,-1]
model.alpha_shift = alpha
model.I = 2*Ic
coeff = "2*I_c"
sol = solve_ivp(f, (0, 10*model.T_ini), y0, method='BDF',
                rtol=1e-7, atol=1e-9,
                t_eval=np.linspace(0, 10*model.T_ini, 200))


fig = plt.figure(figsize=(6, 4))
line, = plt.plot(z_centers, sol.y[:, 0], label=rf"$\alpha = \dfrac{{\pi}}{{{denom}}},\ I={coeff}$")
plt.xlim(z_centers[0], z_centers[-1])
plt.ylim(0.8 * np.min(sol.y), np.max(sol.y) * 1.1)
plt.xlabel('z')
plt.ylabel(r'$\rho(z)$')
plt.title(rf'$I={coeff}$')
plt.legend(loc='upper right')

fig.suptitle(rf'Evolution of $\rho(z,t)$ for different values of $I$ with $\alpha = \dfrac{{\pi}}{{{denom}}}, I_c = 2 \times sec(\alpha)$', fontsize=16)
fig.tight_layout()


def update(frame):
    line.set_ydata(sol.y[:, frame])
    plt.title(rf'$I={coeff},\ t={sol.t[frame]:.2f}$')

    return line,

ani = FuncAnimation(fig, update, frames=len(sol.t), blit=True, interval=50) 
ani.save(f'solution_with_kuramoto_potential_alpha_{alpha:.2f}_{coeff}_bdf.gif', writer='imagemagick')
plt.show()



In [ ]:
first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
plt.plot(sol.t, first_moment, color='green', linewidth=1.5, label=r'$\int_{\Omega} z \rho(z,t)dz $')
# plt.plot(sol.t[0:], np.mean(sol.y, axis=0)[0:], color='blue', linewidth=1.5)
plt.xlabel('t', fontsize=18)
plt.ylabel(r'$\int_{\Omega} z \rho(z,t)dz $', fontsize=14)
# ax.set_ylabel(r'$\max_z ( \rho(z,t) )$', fontsize=14)
plt.title(rf'$I={coeff}$')
plt.xlim(10, 18)

    # ax.set_xlim([15,20])
# plt.ylim([10, np.max(sol.y)*1.1])
plt.legend()
# plt.suptitle(r'Periodicity evolution of the density with $\alpha = {alpha},\quad I_c = 2 \times sec(\alpha)$', fontsize=16)
plt.tight_layout()

fig = plt.figure(figsize=(12/2.54, 8/2.54), dpi=200)
plt.imshow(
    sol.y,
    aspect='auto',
    cmap='Blues',
    origin='lower',
    extent=(
        float(np.min(sol.t)),
        float(np.max(sol.t)),
        float(np.min(z_centers)),
        float(np.max(z_centers)),
    )
)

plt.colorbar(label=r'$\rho(t,z)$')
plt.xlabel("t", fontsize = 14)
plt.ylabel('z', fontsize=14)
# plt.ylim(-10,7)
# plt.xlim(10, 18)
# ax[0].set_title(f'Density $\rho(t,z) at I = {I_vals[ind]:.4f}')
plt.tight_layout()
plt.savefig(rf'surface_map_density_alpha_{alpha}_{coeff}.png')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 4), sharex=True, sharey=True)

for sol, I, coeff, ax in zip(sols, I_values, coeff_labels, axes):
    first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
    ax.plot(sol.t, first_moment, color='green', linewidth=1.5, label=r'$\int_{\Omega} z \rho(z,t)dz $')
    # ax.plot(sol.t[0:], np.max(sol.y, axis=0)[0:], color='blue', linewidth=1.5)
    ax.set_xlabel('t', fontsize=18)
    ax.set_ylabel(r'$\int_{\Omega} z \rho(z,t)dz $', fontsize=14)
    # ax.set_ylabel(r'$\max_z ( \rho(z,t) )$', fontsize=14)
    ax.set_title(rf'$I={coeff}$')

    # ax.set_xlim([15,20])
# plt.ylim([10, np.max(sol.y)*1.1])
    ax.legend()
plt.suptitle(r'Periodicity evolution of the density with $\alpha = {alpha},\quad I_c = 2 \times sec(\alpha)$', fontsize=16)
plt.tight_layout()
plt.savefig(f'solution_first_moment_with_kuramoto_potential_alpha_{alpha:.2f}_bdf.png')
plt.show()

In [ ]:
def run(model,f, J,n_z,orbit_method,p0,T, y0, filename=None, t_scale=True):
    epsilon = model.precision
    model.n_z = n_z
    model.p0 = p0 #Size of the dominant subspace
    model.update_params()#(**{'n_z': n_z}) #Update the model parameters
    #Initialization
    z, z_centers, h = model.mesh1D
    
    T_unit = 1.0
    t_span = (0, 6*T)
    H = h*np.ones_like(y0)
    model.m0 = H @ y0
    print('Mass at initial point:', model.m0)
    #We integrate sufficiently the equation to find a good starting point
    phi_t = solve_ivp(f, t_span, y0, method='BDF', jac = J,
                     rtol=1e-7, atol=1e-9,
                     t_eval= [6*T])#np.linspace(0, 10, 100))
    
    y_T = phi_t.y[:,-1] #Using phi(y0,T0) as a starting point
    
  
    print('Mass at the starting point:', H@y_T)
    orbit_finder = orbit(f,y0,T, J ,2, solve_ivp, model.method, 10000,model.max_iter, epsilon)
    
    V_0 = np.eye(len(y0))[:,:p0+model.pe]#Initial guess of the subspace
    #The arguments to pass to the orbit_finder method
    args_func = {
    "y_0": y_T,
    "T_0": T,
    "model": model,
    "f_unscaled": f,
    "jac_unscaled": J,
    "alpha_0": model.alpha,
    "Max_iter": model.max_iter,
    "epsilon": epsilon,
    "subsp_iter": model.subsp_iter,
    "l": model.picard_iter,
    "Ve_0": V_0,
    "p0": p0,
    "pe": model.pe,
    "rho": model.rho,
    "phase_cond": 2,
    "l": model.picard_iter,
    "full_sub_iter": model.full_sub_iter, # Use the full subspace iteration if True for the subspace iteration with projection
    "h": h
    }
    method_to_call= getattr(orbit_finder, orbit_method)

    return call_method(method_to_call, **args_func)

In [ ]:
nz = model.n_z
model.I = 2*Ic
model.alpha = 0.0
# model.p0 = 10
# sol = sols[-2]
y0 = sol.y[:,-1]+0.005*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
T = 2.4
model.precision = 1e-4
k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged, mass = run(model,f,J,nz,"Newton_mass_conserv4",
                                                                         model.p0,T,y0, filename=None, t_scale=True);

#save the results to a pickle file
# with open('./Results/mckean_vlasov_param_1.in/Np_spence_18_03_2026.pkl', 'wb') as file:
#     data = {
#         'k': k,
#         'T_by_iter': T_by_iter,
#         'y_by_iter': y_by_iter,
#         'Norm_B': Norm_B,
#         'Abs_Err': Abs_Err,
#         'Rel_Err': Rel_Err,
#         'converged': converged,                
#         'Delta_mass': mass,
#         'p0': model.p0,
#         'alpha': model.alpha,
#         'n_z': model.n_z
#     }
#     pickle.dump(data, file)

In [ ]:
def circulant_from_fft(h):
    n = len(h)
    C = np.column_stack([
        np.fft.ifft(np.fft.fft(h) * np.fft.fft(np.eye(n)[:, j])).real
        for j in range(n)
    ])
    return C

K = np.array([1, 2, 3, 4], dtype=float)
C = circulant_from_fft(K)
print(C)

In [ ]:
np.linalg.norm(sp.linalg.circulant(K) - circulant_from_fft(K))

In [ ]:
%%timeit
circulant_from_fft(K)
#Let see time differences against the circulant matrix from sp.linalg

In [ ]:
%%timeit
sp.linalg.circulant(K)

In [ ]:
#Using fixed point iteration to find the stationary solution
y_fp = (1/(2*np.pi))*np.ones_like(z_centers) + 0.000001*np.sin(z_centers)
# y_fp = sol.y[:, -1] #use the last time point as an initial guess for the fixed point iteration
print(sp.integrate.trapezoid(y_fp, z_centers))
from scipy.optimize import fixed_point
def fixed_point_func(y):
    normalizer = sp.integrate.trapezoid(np.exp(-model.V_kuramoto(z_centers,y)), z_centers)
    return np.exp(-model.V_kuramoto(z_centers,y))/normalizer

y_fp = fixed_point(fixed_point_func, y_fp,method='del2', xtol=1e-7, maxiter=90000)
plt.plot(z_centers, y_fp, color='red', linewidth=1.5, label=r'Fixed Point Solution')
plt.xlabel('z',fontsize =18)
plt.ylabel(r'$\rho(z)$', fontsize = 18)
plt.title('Fixed Point Solution of the Density')
plt.legend()
plt.grid()
plt.show()

sp.integrate.trapezoid(y_fp, z_centers)-1

In [ ]:
model.n_z = 10
model.update_params()
rho =  np.random.rand(model.n_z-1)
model.I = 1.5
Jac = model.jacobian_per(0, rho)
Jac_I = model.df_dI_per(0, rho)
f = model.dydt_per

def compute_jacobian(f, x, epsi=1e-5):
    """    Compute the Jacobian of a vector function f at point x using finite differences.
    """

    #Do not use explilicite loop to compute the Jacobian
    n = len(x)
    m = len(f(0,x))
    J = np.zeros((m, n))

    for i in range(m):
        for j in range(n):
            x_plus = np.copy(x)
            x_minus = np.copy(x)
            x_plus[j] += epsi
            x_minus[j] -= epsi
            J[i, j] = (f(0,x_plus)[i] - f(0,x_minus)[i]) / (2 * epsi)
      
    return J

J_num = compute_jacobian(f,rho, epsi=1e-5)
# J_num_I = df_dI(f, rho, model.I, epsi=1e-5)
#Compare the two Jacobians
print('Difference between analytical and numerical Jacobian:', np.linalg.norm(Jac - J_num))
#Check that the Jacobian is correct using the definition

# model.jacobian(0, rho) @ (rho) - f(0, rho)
# print(Jac @ (rho) - f(0, rho))

In [ ]:
mask = np.abs(Jac) > 1e-10
plt.spy(Jac*mask, markersize=10)
plt.show()

# plt.spy(J_nonlin, markersize=10)
mask = np.abs(J_num) > 1e-10
plt.spy(J_num*mask, markersize=10)


In [ ]:
#Checking the jacobian function
R = []
H = [1e-1,1e-2,1e-3,1e-4,1e-5,1e-6,1e-7,1e-8]
for eps in H:
    r = f(0, rho + eps*rho) - f(0, rho)  - Jac @ (eps*rho)
    R.append(np.linalg.norm(r))
print("Residuals for different epsilons:", R)

plt.figure(figsize=(10, 6))
plt.plot(H, R, marker='o')
#plot the slope h***2
plt.plot(H, [R[0] * (h / H[0])**2 for h in H], marker='*',linestyle='--', color='red', label='Slope ~ h^2')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Epsilon (h)')
plt.ylabel('Residual Norm')
plt.legend()
plt.title('Residual Norm vs Epsilon')
plt.grid(True)